In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
print(catalog_name)
print(schema_name)
print(bronze_prefix)
print(landing_folder_path)

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST CIRCUITS FILE
# ============================================================
# Purpose:
# 1. Read circuits.csv using Spark DataFrame Reader API
# 2. Apply an explicit schema
# 3. Add ingestion metadata
# 4. Write the data into a Bronze Delta table
# ============================================================


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# Source file location

source_file = f"{landing_folder_path}/circuits.csv"

# Target Bronze table
# We use default schema because separate Bronze schema access
# is not available.

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_circuits"
)

print("Source File :", source_file)
print("Target Table:", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define schema for circuits.csv
# ============================================================

from pyspark.sql.types import (StructType,StructField,StringType,DoubleType)

circuits_schema = StructType([
    StructField("circuitId", StringType(), True),
    StructField("url", StringType(), True),
    StructField("circuitName", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("locality", StringType(), True),
    StructField("country", StringType(), True)
])


# COMMAND ----------

# ============================================================
# Step 2 - Read circuits.csv
# ============================================================

circuits_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(circuits_schema)
    .load(source_file)
)

display(circuits_df)


# COMMAND ----------

# Check DataFrame schema

circuits_df.printSchema()


# COMMAND ----------

# ============================================================
# Step 3 - Add ingestion metadata
# ============================================================

circuits_final_df = add_ingestion_metadata(circuits_df)

display(circuits_final_df)


# COMMAND ----------

# ============================================================
# Step 4 - Write to Bronze Delta table
# ============================================================

(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 5 - Validate Bronze Delta table
# ============================================================

display(
    spark.table(table_name)
)